### **Environment Setup and Library Imports**

This section initializes the computational environment by installing and importing the necessary dependencies. It performs the following key actions:

1.  **Dependency Installation**: Installs external Python packages required for the project, including `bertopic` (for neural topic modeling), `gensim` (for coherence metrics), `plotly` (for interactive visualizations), and `wordcloud` (for visual summaries).
2.  **Library Configuration**: Imports core libraries for data manipulation (`pandas`, `numpy`), machine learning (`sklearn`), and visualization (`matplotlib`, `seaborn`).
3.  **System Integration**: Mounts Google Drive to access the dataset and defines global visualization settings.

In [ ]:
# 1. LIBRARY INSTALLATION
!pip install bertopic gensim plotly wordcloud -q

# 2. LIBRARY IMPORTS

# --- Standard Library & System ---
import os
import re
import gc
import joblib

# --- Data Manipulation & Computation ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from wordcloud import WordCloud
from IPython.display import display, HTML

# --- Machine Learning & NLP (Sklearn, Gensim, BERTopic) ---
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# --- Google Colab Integration ---
from google.colab import drive

# Set a clean visual style for Matplotlib/Seaborn plots
sns.set_style("white")

### **Topic Modeling Binary & Multiclass Analysis**

This section executes the core unsupervised learning phase, performing a comparative analysis between **LSA (Statistical Baseline)** and **BERTopic (Deep Semantic Clustering)** across two distinct granularities:

* **Binary Analysis**: Contrasts macroscopic themes between **Negative** (1-2 Stars) and **Positive** (4-5 Stars) sentiment groups (30k samples/class).
* **Multiclass Analysis**: Tracks the granular evolution of topics across the specific **1 to 5 Star** rating scale (20k samples/class).

To ensure rigorous evaluation, the Coherence Score is calculated strictly against the `topic_text_clean` corpus (lemmatized tokens + bigrams). This validates that the extracted topics map correctly to the pre-processed vocabulary, regardless of the model used.


In [ ]:
# ==============================================================================
# 1. SETUP AND CONFIGURATION
# ==============================================================================
SAMPLES_PER_CLASS = 30000

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Path Configuration
BASE_PATH = '/content/drive/MyDrive/MAGISTRALE/Text_Mining'
DATA_PATH = os.path.join(BASE_PATH, 'Datasets')
VECTORS_PATH = os.path.join(DATA_PATH, 'vectors')
RESULTS_PATH = os.path.join(BASE_PATH, 'Results', 'results_topic_modeling_binario')
os.makedirs(RESULTS_PATH, exist_ok=True)

# ==============================================================================
# 2. DATA LOADING
# ==============================================================================
print("[INFO] Loading Training Data...")
# Load 'topic_text_clean' which matches the TF-IDF vocabulary
df_train = pd.read_csv(os.path.join(DATA_PATH, 'train_dataset.csv'),
                       usecols=['text_review', 'stars_review', 'topic_text_clean'])

# Data cleaning and type casting
df_train['text_review'] = df_train['text_review'].fillna("")
df_train['topic_text_clean'] = df_train['topic_text_clean'].fillna("")
df_train['stars_review'] = pd.to_numeric(df_train['stars_review'], errors='coerce')
df_train = df_train.dropna(subset=['stars_review'])
df_train['stars_review'] = df_train['stars_review'].astype(int)

# Load Vectors
print("[INFO] Loading Vector Representations...")
tfidf_matrix = joblib.load(os.path.join(VECTORS_PATH, 'tfidf_topic_train.pkl'))
tfidf_vectorizer = joblib.load(os.path.join(VECTORS_PATH, 'tfidf_topic_vectorizer.pkl'))
feature_names = tfidf_vectorizer.get_feature_names_out()
bert_embeddings = np.load(os.path.join(VECTORS_PATH, 'bert_train.npy'), mmap_mode='r')

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================
def get_coherence(topic_words_list, docs_tokens, local_dictionary):
    """
    Calculates the C_v Coherence Score using pre-processed tokens.
    """
    if not topic_words_list: return 0.0
    try:
        cm = CoherenceModel(
            topics=topic_words_list,
            texts=docs_tokens,
            dictionary=local_dictionary,
            coherence='c_v'
        )
        score = cm.get_coherence()
        if np.isnan(score): return 0.0
        return score
    except Exception as e:
        print(f"   [WARNING] Coherence calculation failed: {e}")
        return 0.0

def get_diversity(topic_words_list):
    """Calculates Topic Diversity."""
    if not topic_words_list: return 0.0
    unique_words = set()
    total_words = 0
    for words in topic_words_list:
        unique_words.update(words)
        total_words += len(words)
    if total_words == 0: return 0.0
    return len(unique_words) / total_words

# ==============================================================================
# 4. STRATIFIED ANALYSIS PIPELINE
# ==============================================================================
def analyze_group_stratified(group_name, target_stars):
    print(f"\n{'='*60}")
    print(f"ANALYSIS GROUP: {group_name} (Stratified)")
    print(f"{'='*60}")

    # A. Sampling
    collected_indices = []
    for star in target_stars:
        star_indices = df_train[df_train['stars_review'] == star].index.to_numpy()
        n_take = min(len(star_indices), SAMPLES_PER_CLASS)
        if n_take > 0:
            picked = np.random.choice(star_indices, size=n_take, replace=False)
            collected_indices.extend(picked)

    final_indices = np.array(collected_indices)
    np.random.shuffle(final_indices)

    if len(final_indices) < 50:
        print("[WARNING] Not enough data. Skipping.")
        return

    print(f"   > Total Documents: {len(final_indices)}")

    # B. Tokenization (Using 'topic_text_clean')
    print("   > Generating tokens from 'topic_text_clean'...")
    current_docs_clean = df_train.iloc[final_indices]['topic_text_clean'].tolist()
    tokens_subset = [str(doc).split() for doc in current_docs_clean]
    local_dictionary = Dictionary(tokens_subset)

    # C. LSA Analysis
    print("   > Running LSA...")
    tfidf_sub = tfidf_matrix[final_indices]
    lsa = TruncatedSVD(n_components=5, random_state=42)
    lsa.fit(tfidf_sub)

    lsa_csv_data = []
    lsa_topic_words = []

    for i, comp in enumerate(lsa.components_):
        top_idx = comp.argsort()[:-11:-1]
        top_words = [feature_names[k] for k in top_idx]
        lsa_topic_words.append(top_words)
        lsa_csv_data.append({'Topic': i, 'Keywords': ", ".join(top_words)})

    lsa_coh = get_coherence(lsa_topic_words, tokens_subset, local_dictionary)
    lsa_div = get_diversity(lsa_topic_words)
    print(f"      LSA Metrics -> Coherence: {lsa_coh:.4f} | Diversity: {lsa_div:.4f}")

    pd.DataFrame(lsa_csv_data).to_csv(os.path.join(RESULTS_PATH, f'lsa_{group_name}.csv'), index=False)
    del tfidf_sub, lsa, lsa_csv_data
    gc.collect()

    # D. BERTopic Analysis
    print("   > Running BERTopic...")
    embs_train = bert_embeddings[final_indices]

    # Docs for BERTopic training (Raw Text)
    docs_raw = df_train.iloc[final_indices]['text_review'].tolist()

    try:
        topic_model = BERTopic(
            language="english",
            verbose=False,
            min_topic_size=max(20, int(len(final_indices)/300)),
            # Use standard stop words for the visual representation
            vectorizer_model=CountVectorizer(stop_words="english", min_df=10),
            calculate_probabilities=False,
            low_memory=True
        )

        topic_model.fit_transform(docs_raw, embs_train)

        bert_topic_words = []
        topics = topic_model.get_topics()

        has_valid_topics = False
        for topic_id in topics:
            if topic_id == -1: continue
            words = [word for word, _ in topics[topic_id]]
            if words:
                bert_topic_words.append(words)
                has_valid_topics = True

        if not has_valid_topics:
            bert_coh, bert_div = 0.0, 0.0
        else:
            bert_coh = get_coherence(bert_topic_words, tokens_subset, local_dictionary)
            bert_div = get_diversity(bert_topic_words)

        print(f"      BERTopic Metrics -> Coherence: {bert_coh:.4f} | Diversity: {bert_div:.4f}")

        # Save CSV Results ONLY (No HTML visualization)
        topic_model.get_topic_info().to_csv(os.path.join(RESULTS_PATH, f'bertopic_{group_name}.csv'), index=False)

    except Exception as e:
        print(f"     [ERROR] BERTopic failed: {e}")

    # Cleanup
    del tokens_subset, local_dictionary, current_docs_clean, docs_raw
    gc.collect()

# ==============================================================================
# 5. EXECUTION LOOP
# ==============================================================================
sentiment_groups = {
    'NEGATIVE': [1, 2],
    'POSITIVE': [4, 5]
}

for name, stars in sentiment_groups.items():
    analyze_group_stratified(name, stars)

print(f"\n[SUCCESS] Binary Topic Modeling completed. Results saved in: {RESULTS_PATH}")

In [ ]:
# ==============================================================================
# 1. SETUP AND CONFIGURATION
# ==============================================================================
SAMPLES_PER_CLASS = 20000

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Path Configuration
BASE_PATH = '/content/drive/MyDrive/MAGISTRALE/Text_Mining'
DATA_PATH = os.path.join(BASE_PATH, 'Datasets')
VECTORS_PATH = os.path.join(DATA_PATH, 'vectors')
RESULTS_PATH = os.path.join(BASE_PATH, 'Results', 'results_topic_modeling_multiclasse')
os.makedirs(RESULTS_PATH, exist_ok=True)

# ==============================================================================
# 2. DATA LOADING
# ==============================================================================
print("[INFO] Loading Training Data...")
# Load 'topic_text_clean' for consistent metric calculation
df_train = pd.read_csv(os.path.join(DATA_PATH, 'train_dataset.csv'),
                       usecols=['text_review', 'stars_review', 'topic_text_clean'])

# Data Cleaning
df_train['text_review'] = df_train['text_review'].fillna("")
df_train['topic_text_clean'] = df_train['topic_text_clean'].fillna("")
df_train['stars_review'] = pd.to_numeric(df_train['stars_review'], errors='coerce')
df_train = df_train.dropna(subset=['stars_review'])
df_train['stars_review'] = df_train['stars_review'].astype(int)

# Load Vectors
print("[INFO] Loading Vector Representations...")
tfidf_matrix = joblib.load(os.path.join(VECTORS_PATH, 'tfidf_topic_train.pkl'))
tfidf_vectorizer = joblib.load(os.path.join(VECTORS_PATH, 'tfidf_topic_vectorizer.pkl'))
feature_names = tfidf_vectorizer.get_feature_names_out()
bert_embeddings = np.load(os.path.join(VECTORS_PATH, 'bert_train.npy'), mmap_mode='r')

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================
def get_coherence(topic_words_list, docs_tokens, local_dictionary):
    """
    Calculates C_v Coherence using the batch-specific dictionary.
    """
    if not topic_words_list: return 0.0
    try:
        cm = CoherenceModel(
            topics=topic_words_list,
            texts=docs_tokens,
            dictionary=local_dictionary,
            coherence='c_v'
        )
        score = cm.get_coherence()
        if np.isnan(score): return 0.0
        return score
    except Exception as e:
        print(f"   [WARNING] Coherence calculation failed: {e}")
        return 0.0

def get_diversity(topic_words_list):
    """
    Calculates Topic Diversity score (ratio of unique words).
    """
    if not topic_words_list: return 0.0
    unique_words = set()
    total_words = 0
    for words in topic_words_list:
        unique_words.update(words)
        total_words += len(words)
    if total_words == 0: return 0.0
    return len(unique_words) / total_words

# ==============================================================================
# 4. MULTICLASS ANALYSIS PIPELINE
# ==============================================================================
def analyze_single_class(star):
    group_name = f"STAR_{star}"
    print(f"\n{'='*60}")
    print(f"ANALYSIS CLASS: {star} STARS")
    print(f"{'='*60}")

    # A. Sampling
    indices = df_train[df_train['stars_review'] == star].index.to_numpy()
    if len(indices) < 50:
        print("[WARNING] Insufficient data. Skipping.")
        return

    n_take = min(len(indices), SAMPLES_PER_CLASS)
    indices_sample = np.random.choice(indices, size=n_take, replace=False)

    # B. Tokenization (Using 'topic_text_clean')
    print(f"   > Processing batch of {n_take} documents...")
    # We use the pre-cleaned text for metric validation
    current_docs_clean = df_train.iloc[indices_sample]['topic_text_clean'].tolist()
    tokens_subset = [str(doc).split() for doc in current_docs_clean]
    local_dictionary = Dictionary(tokens_subset)

    # C. LSA Analysis
    print("   > Running LSA...")
    lsa = TruncatedSVD(n_components=5, random_state=42)
    lsa.fit(tfidf_matrix[indices_sample])

    lsa_csv_data = []
    lsa_topic_words = []
    for i, comp in enumerate(lsa.components_):
        top_idx = comp.argsort()[:-11:-1]
        words = [feature_names[k] for k in top_idx]
        lsa_topic_words.append(words)
        lsa_csv_data.append({'Topic': i, 'Keywords': ", ".join(words)})

    # LSA Metrics
    lsa_coh = get_coherence(lsa_topic_words, tokens_subset, local_dictionary)
    lsa_div = get_diversity(lsa_topic_words)
    print(f"     LSA Metrics -> Coherence: {lsa_coh:.4f} | Diversity: {lsa_div:.4f}")

    # Save LSA
    pd.DataFrame(lsa_csv_data).to_csv(os.path.join(RESULTS_PATH, f'lsa_{group_name}.csv'), index=False)
    del lsa, lsa_csv_data
    gc.collect()

    # D. BERTopic Analysis
    print("   > Running BERTopic...")
    try:
        topic_model = BERTopic(
            language="english",
            verbose=False,
            min_topic_size=max(10, int(len(indices_sample)/200)),
            # Use standard stopwords for visualization
            vectorizer_model=CountVectorizer(stop_words="english", min_df=5),
            calculate_probabilities=False,
            low_memory=True
        )

        # We use Pre-computed Embeddings + Raw Text for the model fitting
        embs_sub = bert_embeddings[indices_sample]
        docs_raw = df_train.iloc[indices_sample]['text_review'].tolist()

        topic_model.fit_transform(docs_raw, embs_sub)

        # Extract words for metrics
        bert_topic_words = []
        topics = topic_model.get_topics()
        for topic_id in topics:
            if topic_id == -1: continue
            words = [word for word, _ in topics[topic_id]]
            if words: bert_topic_words.append(words)

        # BERTopic Metrics (Validated against Clean Text)
        bert_coh = get_coherence(bert_topic_words, tokens_subset, local_dictionary)
        bert_div = get_diversity(bert_topic_words)
        print(f"     BERTopic Metrics -> Coherence: {bert_coh:.4f} | Diversity: {bert_div:.4f}")

        # Save BERTopic Results (CSV ONLY)
        topic_model.get_topic_info().to_csv(os.path.join(RESULTS_PATH, f'bertopic_{group_name}.csv'), index=False)

    except Exception as e:
        print(f"     [ERROR] BERTopic failed: {e}")

    # Final Cleanup for Batch
    del tokens_subset, local_dictionary, current_docs_clean, docs_raw
    gc.collect()

# ==============================================================================
# 5. EXECUTION LOOP
# ==============================================================================
for star in [1, 2, 3, 4, 5]:
    analyze_single_class(star)

print(f"\n[SUCCESS] Multiclass Topic Modeling completed. Results saved in: {RESULTS_PATH}")

### **Visualizations**

This module transforms the numerical results of LSA and BERTopic into visual insights, generating two types of output stored in `/Results`.

**1. Interactive Hierarchical Charts (Plotly)**
* **Binary Treemap**: Combines Positive and Negative datasets into a nested, interactive rectangular chart to compare sentiment volume and topic distribution.
* **Multiclass Sunburst**: Visualizes the topic hierarchy across the 5-star scale using a concentric ring chart, enabling the exploration of topic evolution per rating.

**2. Static Semantic Visualizations (Matplotlib & WordCloud)**
* **LSA Keyword Bars**: Generates ranked bar charts for the top latent topics, visualizing the most significant keywords for each sentiment group.
* **Semantic WordClouds**: Creates intuitive visual summaries (WordClouds) by aggregating keyword weights from the LSA matrices, highlighting the dominant vocabulary for both Binary and Multiclass analyses.

In [ ]:
# ==============================================================================
# 1. SETUP AND PATH CONFIGURATION
# ==============================================================================
# Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Base Directory Configuration
BASE_DIR = '/content/drive/MyDrive/MAGISTRALE/Text_Mining'

# Results Directory (Artifact Retrieval)
RESULTS_PATH = os.path.join(BASE_DIR, 'Results')

# Specific Sub-directories for reading
PATH_BINARY = os.path.join(RESULTS_PATH, 'results_topic_modeling_binario')
PATH_MULTICLASS = os.path.join(RESULTS_PATH, 'results_topic_modeling_multiclasse')

# ==============================================================================
# 2. VISUALIZATION UTILITY FUNCTION
# ==============================================================================
def display_modeling_results(folder_path, group_name, title):
    """
    Retrieves and displays LSA keywords and BERTopic interactive visualizations
    for a specific analysis group.

    Parameters:
        folder_path (str): Path to the directory containing the results.
        group_name (str): Identifier for the group (e.g., 'POSITIVE', 'STAR_1').
        title (str): Display title for the section.
    """
    print(f"\n{'#'*80}")
    print(f"VISUALIZATION: {title}")
    print(f"{'#'*80}")

    # ---------------------------------------------------------
    # A. Display LSA Results (Table)
    # ---------------------------------------------------------
    lsa_file = os.path.join(folder_path, f'lsa_{group_name}.csv')

    if os.path.exists(lsa_file):
        print(f"\n[LSA] Top 5 Latent Topics:")
        try:
            df_lsa = pd.read_csv(lsa_file)
            # Iterate and print formatted keywords for readability
            for i, row in df_lsa.head(5).iterrows(): # Show only top 5 as printed
                print(f"   Topic {row['Topic']}: {row['Keywords']}")
        except Exception as e:
            print(f"   [ERROR] Could not read LSA file: {e}")
    else:
        print(f"   [WARNING] LSA file not found: {lsa_file}")

    # ---------------------------------------------------------
    # B. Display BERTopic Info (Cluster Summary)
    # ---------------------------------------------------------
    bert_file = os.path.join(folder_path, f'bertopic_{group_name}.csv')

    if os.path.exists(bert_file):
        print(f"\n[BERTopic] Top 5 Clusters (by Document Frequency):")
        try:
            df_bert = pd.read_csv(bert_file)

            # Filter for relevant columns to keep display clean
            cols_to_show = ['Topic', 'Count', 'Name', 'Representation']
            cols_final = [c for c in cols_to_show if c in df_bert.columns]

            display(df_bert[cols_final].head(5))

            # Check for outliers (Topic -1)
            outliers = df_bert[df_bert['Topic'] == -1]
            if not outliers.empty:
                n_outliers = outliers.iloc[0]['Count']
                print(f"   [NOTE] Topic -1 (Outliers) contains {n_outliers} unclassified documents.")
        except Exception as e:
            print(f"   [ERROR] Could not read BERTopic file: {e}")
    else:
        print(f"   [WARNING] BERTopic CSV not found: {bert_file}")

    # ---------------------------------------------------------
    # C. Display BERTopic Chart (Read & Render Only)
    # ---------------------------------------------------------
    viz_file = os.path.join(folder_path, f'bertopic_viz_{group_name}.html')

    if os.path.exists(viz_file):
        print(f"\n[BERTopic] Interactive Intertopic Distance Map:")
        try:
            with open(viz_file, 'r', encoding='utf-8') as f:
                html_content = f.read()
            # Render HTML content directly in the notebook output
            display(HTML(html_content))
        except Exception as e:
            print(f"   [ERROR] Could not render HTML: {e}")
    else:
        print(f"   [WARNING] Visualization file not found: {viz_file}")

# ==============================================================================
# 3. EXECUTION: BINARY ANALYSIS (Positive vs Negative)
# ==============================================================================
print("\n" + "="*40 + " PART 1: BINARY ANALYSIS " + "="*40)

# Negative Group (1-2 Stars)
display_modeling_results(PATH_BINARY, 'NEGATIVE', 'Negative Sentiment Group')

# Positive Group (4-5 Stars)
display_modeling_results(PATH_BINARY, 'POSITIVE', 'Positive Sentiment Group')

# ==============================================================================
# 4. EXECUTION: MULTICLASS ANALYSIS (1-5 Stars)
# ==============================================================================
print("\n" + "="*40 + " PART 2: MULTICLASS ANALYSIS " + "="*40)

for star in [1, 2, 3, 4, 5]:
    group_id = f"STAR_{star}"
    display_modeling_results(PATH_MULTICLASS, group_id, f'Class: {star} STARS')

In [ ]:
# ==============================================================================
# 1. SETUP AND PATH CONFIGURATION
# ==============================================================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/MAGISTRALE/Text_Mining'
RESULTS_PATH = os.path.join(BASE_DIR, 'Results')

# Input Paths
PATH_BINARY = os.path.join(RESULTS_PATH, 'results_topic_modeling_binario')
PATH_MULTICLASS = os.path.join(RESULTS_PATH, 'results_topic_modeling_multiclasse')

# Output Path
IMG_PATH = os.path.join(RESULTS_PATH, 'images')
os.makedirs(IMG_PATH, exist_ok=True)

# ==============================================================================
# 2. HELPER: DATA LOADER
# ==============================================================================
def load_and_prep_data(folder_path, group_name, label_col_name, label_value):
    """
    Loads a BERTopic CSV, removes outliers, and adds a category label
    (e.g., Sentiment='Positive' or Rating='5 Stars').
    """
    csv_path = os.path.join(folder_path, f'bertopic_{group_name}.csv')
    if not os.path.exists(csv_path):
        print(f"   [WARNING] Missing file: {csv_path}")
        return None

    df = pd.read_csv(csv_path)
    # Remove outliers (Topic -1)
    df = df[df['Topic'] != -1].copy()

    if df.empty: return None

    # Create readable labels
    df['Short_Label'] = df['Name'].apply(lambda x: " ".join(str(x).split('_')[1:4]))

    # Add the grouping column (e.g., Sentiment or Rating)
    df[label_col_name] = label_value
    return df

# ==============================================================================
# 3. PLOT 1: COMBINED BINARY TREEMAP
# ==============================================================================
def generate_combined_binary_treemap():
    print("\n>>> GENERATING: Interactive_Treemap_Binary.html")

    # 1. Load and Tag Data
    df_neg = load_and_prep_data(PATH_BINARY, 'NEGATIVE', 'Sentiment', 'Negative')
    df_pos = load_and_prep_data(PATH_BINARY, 'POSITIVE', 'Sentiment', 'Positive')

    if df_neg is None or df_pos is None:
        print("[ERROR] Could not load binary datasets.")
        return

    # 2. Combine
    df_binary = pd.concat([df_neg, df_pos], ignore_index=True)

    # 3. Generate Treemap
    fig = px.treemap(
        df_binary,
        path=[px.Constant("Binary Analysis"), 'Sentiment', 'Short_Label'],
        values='Count',
        title='<b>Binary Topic Distribution (Positive vs. Negative)</b>',
        hover_data=['Name'],
        color='Sentiment',
        color_discrete_map={'Negative': '#d73027', 'Positive': '#1a9850'} # Red/Green
    )

    fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))

    # 4. Save and Show
    save_path = os.path.join(IMG_PATH, 'Interactive_Treemap_Binary.html')
    fig.write_html(save_path)
    print(f"   -> Saved to: {save_path}")
    fig.show()

# ==============================================================================
# 4. PLOT 2: COMBINED MULTICLASS SUNBURST
# ==============================================================================
def generate_combined_multiclass_sunburst():
    print("\n>>> GENERATING: Interactive_Sunburst_Multiclass.html")

    frames = []
    # 1. Load All Stars
    for star in [1, 2, 3, 4, 5]:
        df = load_and_prep_data(PATH_MULTICLASS, f'STAR_{star}', 'Rating', f'{star} Stars')
        if df is not None:
            frames.append(df)

    if not frames:
        print("[ERROR] Could not load multiclass datasets.")
        return

    # 2. Combine
    df_multi = pd.concat(frames, ignore_index=True)

    # 3. Generate Sunburst (Hierarchy: Rating -> Topic)
    fig = px.sunburst(
        df_multi,
        path=['Rating', 'Short_Label'],
        values='Count',
        title='<b>Multiclass Topic Evolution (1-5 Stars)</b>',
        hover_data=['Name'],
        color='Rating',
        # Sequential color scale to represent progression
        color_discrete_map={
            '1 Stars': '#d73027',
            '2 Stars': '#fc8d59',
            '3 Stars': '#fee08b',
            '4 Stars': '#d9ef8b',
            '5 Stars': '#1a9850'
        }
    )

    fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))

    # 4. Save and Show
    save_path = os.path.join(IMG_PATH, 'Interactive_Sunburst_Multiclass.html')
    fig.write_html(save_path)
    print(f"   -> Saved to: {save_path}")
    fig.show()

# ==============================================================================
# 5. EXECUTION
# ==============================================================================
generate_combined_binary_treemap()
generate_combined_multiclass_sunburst()

print(f"\n[SUCCESS] Charts available in: {IMG_PATH}")

In [ ]:
# ==============================================================================
# 1. SETUP AND PATH CONFIGURATION
# ==============================================================================
# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Base Paths
BASE_DIR = '/content/drive/MyDrive/MAGISTRALE/Text_Mining'
RESULTS_DIR = os.path.join(BASE_DIR, 'Results')

# Input Paths (Where LSA CSVs are stored)
PATH_BINARY = os.path.join(RESULTS_DIR, 'results_topic_modeling_binario')
PATH_MULTICLASS = os.path.join(RESULTS_DIR, 'results_topic_modeling_multiclasse')

# Output Path (Where images will be saved)
IMG_PATH = os.path.join(RESULTS_DIR, 'images')
os.makedirs(IMG_PATH, exist_ok=True)

# Visualization Style
sns.set_style("white")

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================
def get_word_freqs_from_lsa(file_path, top_n_topics=3):
    """
    Reads an LSA CSV file and aggregates keyword weights for WordCloud generation.
    It uses a rank-based weighting heuristic (earlier words = higher weight).
    """
    if not os.path.exists(file_path):
        print(f"[WARNING] File not found: {file_path}")
        return None

    try:
        df = pd.read_csv(file_path)
        word_freq = {}

        # Aggregate top N topics
        # Topic 0 is usually the dominant/general topic
        for i in range(min(top_n_topics, len(df))):
            row = df.iloc[i]
            if pd.isna(row['Keywords']): continue

            keywords = str(row['Keywords']).split(', ')

            # Assign importance: Topic 0 > Topic 1 > Topic 2
            topic_importance = 100 - (i * 20)

            for idx, word in enumerate(keywords):
                # Word weight decreases within the topic list
                word_weight = max(1, topic_importance - (idx * 5))
                if word in word_freq:
                    word_freq[word] += word_weight
                else:
                    word_freq[word] = word_weight
        return word_freq
    except Exception as e:
        print(f"[ERROR] Reading {file_path}: {e}")
        return None

# ==============================================================================
# 3. PART A: BAR CHARTS (TOPIC KEYWORDS)
# ==============================================================================
def plot_lsa_bars_binary():
    print("[INFO] Generating Binary Bar Charts (Top 3 Topics)...")

    for sentiment, color_palette in [('NEGATIVE', 'Reds_r'), ('POSITIVE', 'Greens_r')]:
        file_path = os.path.join(PATH_BINARY, f'lsa_{sentiment}.csv')
        if not os.path.exists(file_path):
            print(f"   [SKIP] {sentiment} file missing.")
            continue

        df = pd.read_csv(file_path)
        n_topics = min(3, len(df))

        # Dynamic figure size
        fig, axes = plt.subplots(n_topics, 1, figsize=(10, 3 * n_topics))
        fig.suptitle(f'LSA Latent Topics: {sentiment}', fontsize=16, y=1.02)

        if n_topics == 1: axes = [axes]

        for i in range(n_topics):
            # Extract keywords
            raw_keys = str(df.iloc[i]['Keywords'])
            keywords = raw_keys.split(', ')[:10] # Top 10 words

            # Dummy weights for visualization (descending bar length)
            weights = list(range(len(keywords), 0, -1))

            # Plot
            sns.barplot(x=weights, y=keywords, ax=axes[i], palette=color_palette, orient='h')

            # Formatting
            axes[i].set_title(f"Topic {i} (Dominant Words)", fontsize=12, loc='left', fontweight='bold')
            sns.despine(left=True, bottom=True, ax=axes[i])
            axes[i].set_xticks([]) # Remove x-axis numbers
            axes[i].tick_params(axis='y', labelsize=11, length=0)
            axes[i].set_ylabel("")

        plt.tight_layout()
        save_path = os.path.join(IMG_PATH, f'LSA_Bars_{sentiment}.png')
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.show()
        plt.close()
        print(f"   -> Saved: {save_path}")

def plot_lsa_bars_multiclass_comparison():
    print("[INFO] Generating Multiclass Evolution Chart (1-5 Stars)...")

    fig, axes = plt.subplots(5, 1, figsize=(10, 18))
    fig.suptitle('LSA Semantic Evolution (Dominant Topic per Rating)', fontsize=20, y=0.98)

    # Color scale: Red (1 Star) -> Green (5 Stars)
    colors = ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#1a9850']

    for i, star in enumerate([1, 2, 3, 4, 5]):
        file_path = os.path.join(PATH_MULTICLASS, f'lsa_STAR_{star}.csv')

        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            # Use only Topic 0 (The most representative topic)
            raw_keys = str(df.iloc[0]['Keywords'])
            keywords = raw_keys.split(', ')[:10]
            weights = list(range(len(keywords), 0, -1))

            sns.barplot(x=weights, y=keywords, ax=axes[i], color=colors[i], orient='h')

            axes[i].set_title(f"{star} STARS", fontsize=14, fontweight='bold', loc='left')
            sns.despine(left=True, bottom=True, ax=axes[i])
            axes[i].set_xticks([])
            axes[i].tick_params(axis='y', labelsize=12, length=0)
            axes[i].set_ylabel("")
        else:
            axes[i].text(0.5, 0.5, "Data Not Available", ha='center')
            axes[i].axis('off')

    plt.tight_layout()
    save_path = os.path.join(IMG_PATH, 'LSA_Bars_Multiclass_Comparison.png')
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()
    print(f"   -> Saved: {save_path}")

# ==============================================================================
# 4. PART B: WORDCLOUDS (VISUAL SUMMARY)
# ==============================================================================
def plot_lsa_wordclouds_binary():
    print("[INFO] Generating Binary WordClouds...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    # Negative WordCloud
    freqs_neg = get_word_freqs_from_lsa(os.path.join(PATH_BINARY, 'lsa_NEGATIVE.csv'))
    if freqs_neg and len(freqs_neg) > 0:
        wc = WordCloud(width=800, height=600, background_color='white', colormap='magma').generate_from_frequencies(freqs_neg)
        axes[0].imshow(wc, interpolation='bilinear')
        axes[0].set_title("NEGATIVE Reviews (Macro-Themes)", fontsize=18, color='darkred')
    else:
        axes[0].text(0.5, 0.5, "No Data", ha='center')
    axes[0].axis('off')

    # Positive WordCloud
    freqs_pos = get_word_freqs_from_lsa(os.path.join(PATH_BINARY, 'lsa_POSITIVE.csv'))
    if freqs_pos and len(freqs_pos) > 0:
        wc = WordCloud(width=800, height=600, background_color='white', colormap='viridis').generate_from_frequencies(freqs_pos)
        axes[1].imshow(wc, interpolation='bilinear')
        axes[1].set_title("POSITIVE Reviews (Macro-Themes)", fontsize=18, color='darkgreen')
    else:
        axes[1].text(0.5, 0.5, "No Data", ha='center')
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(IMG_PATH, 'LSA_WordCloud_Binary.png'), bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()

def plot_lsa_wordclouds_multiclass():
    print("[INFO] Generating Multiclass WordClouds...")
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    palettes = ['Reds', 'Oranges', 'YlOrBr', 'Greens', 'BuGn']

    for i, star in enumerate([1, 2, 3, 4, 5]):
        freqs = get_word_freqs_from_lsa(os.path.join(PATH_MULTICLASS, f'lsa_STAR_{star}.csv'))

        if freqs and len(freqs) > 0:
            wc = WordCloud(width=400, height=400, background_color='white', colormap=palettes[i]).generate_from_frequencies(freqs)
            axes[i].imshow(wc, interpolation='bilinear')
            axes[i].set_title(f"{star} STARS", fontsize=14, fontweight='bold')
        else:
            axes[i].text(0.5, 0.5, "N/A", ha='center')

        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(IMG_PATH, 'LSA_WordCloud_Multiclass.png'), bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()

# ==============================================================================
# 5. EXECUTION
# ==============================================================================
plot_lsa_bars_binary()
plot_lsa_bars_multiclass_comparison()
plot_lsa_wordclouds_binary()
plot_lsa_wordclouds_multiclass()

print(f"\n[SUCCESS] All charts generated and saved in: {IMG_PATH}")